In [ ]:
!git clone https://github.com/kratipatel/loan-default-mlbd.git

Cloning into 'loan-default-mlbd'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 91 (delta 33), reused 67 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.33 MiB | 5.10 MiB/s, done.
Resolving deltas: 100% (33/33), done.


In [ ]:
%cd loan-default-mlbd

/content/loan-default-mlbd


In [ ]:
!pip install -q pandas numpy pyarrow scikit-learn pytorch-tabnet

In [ ]:
!python -m src.prepare_data

reading: https://raw.githubusercontent.com/tiwvis/MLBD-Projects/main/data_sets/LoansData_sample.csv.gz
saved: /content/loan-default-mlbd/data/processed/loans_clean.parquet (86141, 71)
        size      mean
split                 
test   12922  0.187587
train  60298  0.187585
val    12921  0.187602


In [ ]:
from src.data import load_splits, make_nn_data
from src.metrics import evaluate

path = "data/processed/loans_clean.parquet"

tr, va, te = load_splits(path)

data, num_cols, cat_cols, cards = make_nn_data(tr, va, te)

print("Train:", len(tr))
print("Validation:", len(va))
print("Test:", len(te))
print("Numerical:", len(num_cols))
print("Categorical:", len(cat_cols))

Train: 60298
Validation: 12921
Test: 12922
Numerical: 59
Categorical: 9


In [ ]:
import torch
from pytorch_tabnet.tab_model import TabNetClassifier

print("PyTorch:", torch.__version__)
print("TabNet installation successful!")

PyTorch: 2.14.1+cpu
TabNet installation successful!


In [ ]:
import numpy as np

In [ ]:
# Separate features and target
X_train_num, X_train_cat, y_train = data["train"]
X_val_num, X_val_cat, y_val = data["val"]
X_test_num, X_test_cat, y_test = data["test"]

# Combine numerical and categorical features
X_train = np.hstack([X_train_num, X_train_cat]).astype(np.float32)
X_val = np.hstack([X_val_num, X_val_cat]).astype(np.float32)
X_test = np.hstack([X_test_num, X_test_cat]).astype(np.float32)

# Identify categorical feature positions
cat_idxs = list(range(len(num_cols), len(num_cols) + len(cat_cols)))
cat_dims = cards

# TabNet expects integer class labels
y_train = y_train.astype(np.int64)
y_val = y_val.astype(np.int64)
y_test = y_test.astype(np.int64)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Categorical indices:", cat_idxs)
print("Categorical dimensions:", cat_dims)

Train: (60298, 68)
Validation: (12921, 68)
Test: (12922, 68)
Categorical indices: [59, 60, 61, 62, 63, 64, 65, 66, 67]
Categorical dimensions: [8, 36, 5, 4, 14, 47, 3, 2, 2]


In [ ]:
# Initial TabNet configuration
tabnet_params = {
    "n_d": 16,
    "n_a": 16,
    "n_steps": 5,
    "gamma": 1.5,
    "lambda_sparse": 1e-4,
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": {"lr": 0.02},
    "mask_type": "sparsemax",
    "seed": 42,
    "verbose": 10
}

model = TabNetClassifier(
    **tabnet_params,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=1
)

print("TabNet model created successfully!")

TabNet model created successfully!


/usr/local/lib/python3.13/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [ ]:
import time

start_time = time.time()

model.fit(
    X_train=X_train,
    y_train=y_train,
    eval_set=[(X_val, y_val)],
    eval_name=["val"],
    eval_metric=["auc"],
    max_epochs=100,
    patience=15,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

train_seconds = time.time() - start_time

print(f"Training time: {train_seconds:.2f} seconds")

epoch 0  | loss: 0.5315  | val_auc: 0.58248 |  0:00:03s
epoch 10 | loss: 0.44497 | val_auc: 0.70201 |  0:00:43s
epoch 20 | loss: 0.43698 | val_auc: 0.71901 |  0:01:23s
epoch 30 | loss: 0.43276 | val_auc: 0.72258 |  0:02:01s
epoch 40 | loss: 0.43279 | val_auc: 0.72319 |  0:02:40s
epoch 50 | loss: 0.43302 | val_auc: 0.66086 |  0:03:20s
epoch 60 | loss: 0.4301  | val_auc: 0.70224 |  0:03:58s
epoch 70 | loss: 0.43191 | val_auc: 0.39689 |  0:04:37s

Early stopping occurred at epoch 76 with best_epoch = 61 and best_val_auc = 0.72796


/usr/local/lib/python3.13/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Training time: 310.15 seconds


In [ ]:
# Generate validation probabilities
val_probs = model.predict_proba(X_val)[:, 1]

# Calculate validation metrics
val_results = evaluate(y_val, val_probs)

print("TabNet Validation Results")
for metric, value in val_results.items():
    print(f"{metric}: {value:.5f}")

TabNet Validation Results
rmse: 0.37441
mae: 0.29490
r2: 0.08021
roc_auc: 0.72796
pr_auc: 0.37640
ks: 0.33096


In [ ]:
import pandas as pd

In [ ]:
# Extract TabNet feature importance
feature_importance = pd.DataFrame({
    "feature": num_cols + cat_cols,
    "importance": model.feature_importances_
})
feature_importance = feature_importance.sort_values(
    "importance", ascending=False
)
display(feature_importance.head(15))

,feature,importance
59,grade,0.455409
26,delinq_amnt,0.120745
11,open_acc,0.119655
24,bc_util,0.054796
31,mort_acc,0.050170
61,home_ownership,0.048321
36,num_actv_rev_tl,0.029581
47,num_tl_op_past_12m,0.024805
6,delinq_2yrs,0.018330
2,int_rate,0.018261


In [ ]:
#save feature importance
import os
os.makedirs("results", exist_ok=True)

feature_importance.to_csv(
    "results/tabnet_feature_importance.csv",
    index=False
)

print("Feature importance saved!")

Feature importance saved!


In [ ]:
# Compare alternative TabNet configurations
configs = {
    "small": {"n_d": 8, "n_a": 8, "n_steps": 3},
    "larger": {"n_d": 32, "n_a": 32, "n_steps": 5}
}

tuning_results = [
    {
        "config": "baseline",
        "val_roc_auc": val_results["roc_auc"],
        "train_seconds": train_seconds
    }
]

best_model = model
best_config = "baseline"
best_auc = val_results["roc_auc"]
best_train_seconds = train_seconds

for name, params in configs.items():
    print(f"\nTraining {name} configuration...")

    candidate = TabNetClassifier(
        **params,
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        cat_emb_dim=1,
        optimizer_fn=torch.optim.Adam,
        optimizer_params={"lr": 0.02},
        seed=42,
        verbose=0
    )

    start = time.time()

    candidate.fit(
        X_train=X_train,
        y_train=y_train,
        eval_set=[(X_val, y_val)],
        eval_name=["val"],
        eval_metric=["auc"],
        max_epochs=60,
        patience=10,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    elapsed = time.time() - start

    val_prob = candidate.predict_proba(X_val)[:, 1]
    auc = evaluate(y_val, val_prob)["roc_auc"]

    tuning_results.append({
        "config": name,
        "val_roc_auc": auc,
        "train_seconds": elapsed
    })

    print(f"Validation ROC-AUC: {auc:.5f}")
    print(f"Training time: {elapsed:.2f}s")

    if auc > best_auc:
        best_auc = auc
        best_model = candidate
        best_config = name
        best_train_seconds = elapsed

print("\nTuning summary:")
display(pd.DataFrame(tuning_results))

print("Best configuration:", best_config)
print("Best validation ROC-AUC:", best_auc)


Training small configuration...

Early stopping occurred at epoch 53 with best_epoch = 43 and best_val_auc = 0.73378


/usr/local/lib/python3.13/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Validation ROC-AUC: 0.73378
Training time: 129.31s

Training larger configuration...

Early stopping occurred at epoch 45 with best_epoch = 35 and best_val_auc = 0.72499


/usr/local/lib/python3.13/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Validation ROC-AUC: 0.72499
Training time: 237.17s

Tuning summary:


,config,val_roc_auc,train_seconds
0,baseline,0.727965,310.148109
1,small,0.733784,129.306795
2,larger,0.724991,237.166181


Best configuration: small
Best validation ROC-AUC: 0.7337843619314774


In [ ]:
print("Selected configuration:", best_config)
print("Best validation AUC:", best_auc)

model = best_model

Selected configuration: small
Best validation AUC: 0.7337843619314774


In [ ]:
# Generate final predictions
val_probs = model.predict_proba(X_val)[:, 1]
test_probs = model.predict_proba(X_test)[:, 1]

# Calculate metrics
val_results = evaluate(y_val, val_probs)
test_results = evaluate(y_test, test_probs)

print("VALIDATION RESULTS")
for k, v in val_results.items():
    print(f"{k}: {v:.5f}")

print("\nTEST RESULTS")
for k, v in test_results.items():
    print(f"{k}: {v:.5f}")

VALIDATION RESULTS
rmse: 0.36772
mae: 0.27995
r2: 0.11279
roc_auc: 0.73378
pr_auc: 0.39571
ks: 0.33536

TEST RESULTS
rmse: 0.36929
mae: 0.28278
r2: 0.10512
roc_auc: 0.72958
pr_auc: 0.38290
ks: 0.34366


In [ ]:
feature_importance = pd.DataFrame({
    "feature": num_cols + cat_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(15))

feature_importance.to_csv(
    "results/tabnet_feature_importance.csv",
    index=False
)

,feature,importance
2,int_rate,0.392798
61,home_ownership,0.129156
52,tot_hi_cred_lim,0.109631
57,term_months,0.089547
8,fico_range_high,0.054327
38,num_bc_tl,0.043775
7,fico_range_low,0.043011
11,open_acc,0.043010
36,num_actv_rev_tl,0.035637
56,credit_hist_years,0.016597


In [ ]:
# Save TabNet results
import json
results_tabnet = {
    "framework": "PyTorch TabNet",
    "params": {
        "n_d": 8,
        "n_a": 8,
        "n_steps": 3,
        "gamma": 1.5,
        "lambda_sparse": 1e-4,
        "learning_rate": 0.02,
        "batch_size": 1024,
        "cat_emb_dim": 1,
        "seed": 42
    },
    "train_seconds": best_train_seconds,
    "val": val_results,
    "test": test_results
}

os.makedirs("reports", exist_ok=True)

with open("reports/results_tabnet.json", "w") as f:
    json.dump(results_tabnet, f, indent=4)

print("TabNet results saved successfully!")

TabNet results saved successfully!


In [ ]:
from google.colab import files

files.download("reports/results_tabnet.json")
files.download("results/tabnet_feature_importance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>